# Module 7: Object-Oriented Programming - Part 3
## Encapsulation, Magic Methods and Inner Classes

### Encapsulation
- **Encapsulation** means keeping an object's data **inside** the object, and controlling how the outside world is allowed to touch it.
- The aim is to stop other code from changing something into a state that makes no sense, such as a rocket with minus fifty litres of fuel.
- Python marks intent with **underscores** in the name:

| Name | Meaning | Can it be reached from outside? |
|---|---|---|
| `name` | **public** | yes, freely |
| `_name` | **protected** | yes, but the underscore is a polite "please do not" |
| `__name` | **private** | not by that name |

- ***Note:*** Python does not truly lock anything away. The single underscore is a **convention between programmers**. The double underscore actually changes the name, as we will see.

In [1]:
class Sensor:
    def __init__(self):
        self.name = "Accelerometer"    # public
        self._location = "Lab 2"       # protected, by convention
        self.__version = "1.0"         # private

s = Sensor()

print(s.name)
print(s._location)                     # works, but you are not supposed to

try:
    print(s.__version)
except AttributeError as e:
    print("AttributeError:", e)

Accelerometer
Lab 2
AttributeError: 'Sensor' object has no attribute '__version'


#### Name Mangling
- A double underscore is not magic. Python simply **renames** the property behind the scenes to `_ClassName__property`. This is called **name mangling**.
- It exists to prevent accidental clashes, not to provide real security.

In [2]:
s = Sensor()

print(s._Sensor__version)              # the real, mangled name
print([a for a in dir(s) if "version" in a])

1.0
['_Sensor__version']


#### Getters and Setters
- If outside code should be able to read or change a private value, we provide **methods** to do it.
- A **getter** returns the value. A **setter** changes it, and can **check** the new value first.
- This is the real point of encapsulation: the object protects its own data.

In [3]:
class Rocket:
    def __init__(self, fuel):
        self.__fuel = fuel

    def get_fuel(self):                # getter
        return self.__fuel

    def set_fuel(self, amount):        # setter, with a check
        if amount >= 0:
            self.__fuel = amount
        else:
            print("Rejected: fuel cannot be negative")

r = Rocket(50)
print(r.get_fuel())

r.set_fuel(120)
print(r.get_fuel())

r.set_fuel(-30)                        # the object protects itself
print(r.get_fuel())

50
120
Rejected: fuel cannot be negative
120


### Magic Methods
- **Magic methods** (also called **dunder methods**, short for double underscore) have names surrounded by two underscores, like `__init__()`.
- You never call them directly. Python calls them for you when you use ordinary syntax.
- You have been using them since Module 2 without knowing. When you write `len("Mars")`, Python calls the string's `__len__()`. When you write `2 + 3`, Python calls `__add__()`.
- By writing these methods in your own class, you make your objects behave like built-in types.

#### __str__ - what print() shows
- Without `__str__()`, printing an object gives an unhelpful line about where it sits in memory.

In [4]:
class Planet:
    def __init__(self, name):
        self.name = name

p = Planet("Mars")
print(type(p).__name__)                # the class name
print(hasattr(p, "__str__"))           # a default exists, but it is not useful

Planet
True


In [5]:
class Planet:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return "Planet: " + self.name

print(Planet("Mars"))                  # print() calls __str__()

Planet: Mars


#### __repr__ - the developer's version
- **`__str__()`** is for a **friendly** message aimed at the user.
- **`__repr__()`** is for an **exact** description aimed at a programmer. It is what you see inside a list, or when you use `repr()`.

In [6]:
class Planet:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return "the planet " + self.name

    def __repr__(self):
        return "Planet('" + self.name + "')"

p = Planet("Mars")

print(p)                               # uses __str__
print(repr(p))                         # uses __repr__
print([p])                             # inside a list, __repr__ is used

the planet Mars
Planet('Mars')
[Planet('Mars')]


#### __eq__ - what == means
- Without `__eq__()`, two separate objects are considered different **even if they hold identical values**.

In [7]:
class Star:
    def __init__(self, name):
        self.name = name

print(Star("Sirius") == Star("Sirius"))    # two different objects

False


In [8]:
class Star:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

print(Star("Sirius") == Star("Sirius"))
print(Star("Sirius") == Star("Vega"))

True
False


#### __add__ - what + means

In [9]:
class Distance:
    def __init__(self, km):
        self.km = km

    def __add__(self, other):
        return Distance(self.km + other.km)

    def __str__(self):
        return str(self.km) + " km"

d1 = Distance(10)
d2 = Distance(5)

print(d1 + d2)                         # Python calls d1.__add__(d2)

15 km


#### __len__ - what len() means

In [10]:
class Fleet:
    def __init__(self, ships):
        self.ships = ships

    def __len__(self):
        return len(self.ships)

print(len(Fleet(["Orion", "Apollo", "Artemis"])))

3


#### __lt__ - what < means, and how sorting works
- **`__lt__()`** stands for "less than". Once Python knows how to compare two of your objects, **`sorted()`** works on them.
- This connects back to the sorting you did with `sorted()` and a lambda in Module 5.

In [11]:
class Star:
    def __init__(self, name, magnitude):
        self.name = name
        self.magnitude = magnitude

    def __lt__(self, other):
        return self.magnitude < other.magnitude

    def __repr__(self):
        return self.name

stars = [Star("Vega", 0.03), Star("Sirius", -1.46), Star("Polaris", 1.98)]

print(Star("Vega", 0.03) < Star("Polaris", 1.98))
print(sorted(stars))                   # brightest first, since smaller is brighter

True
[Sirius, Vega, Polaris]


#### __contains__ - what in means

In [12]:
class Fleet:
    def __init__(self, ships):
        self.ships = ships

    def __contains__(self, item):
        return item in self.ships

f = Fleet(["Orion", "Apollo"])

print("Orion" in f)
print("Zeus" in f)

True
False


#### __call__ - making an object callable
- **`__call__()`** lets you use an object **as if it were a function**.

In [13]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(11))                      # looks like a function call
print(triple(11))
print(callable(double))

22
33
True


#### Summary of Magic Methods
| Method | Python calls it when you write |
|---|---|
| `__init__` | `MyClass(...)` |
| `__str__` | `print(obj)` or `str(obj)` |
| `__repr__` | `repr(obj)`, or the object inside a list |
| `__eq__` | `obj1 == obj2` |
| `__add__` | `obj1 + obj2` |
| `__len__` | `len(obj)` |
| `__lt__` | `obj1 < obj2`, and `sorted()` |
| `__contains__` | `item in obj` |
| `__call__` | `obj(...)` |

### Inner Classes
- A class can be defined **inside another class**. This is called an **inner class** or a nested class.
- It is useful when the inner thing only makes sense as a part of the outer thing, such as an engine belonging to a rocket.

In [14]:
class Rocket:
    def __init__(self, name, engine_count):
        self.name = name
        self.engine = self.Engine(engine_count)    # build the inner object

    class Engine:
        def __init__(self, count):
            self.count = count

        def describe(self):
            print("This rocket has", self.count, "engines")

r = Rocket("Falcon Heavy", 27)

print(r.name)
r.engine.describe()

Falcon Heavy
This rocket has 27 engines


- An inner class can also be reached directly through the outer class name.

In [15]:
e = Rocket.Engine(3)
e.describe()

This rocket has 3 engines


### Putting It All Together
A final example using inheritance, encapsulation and magic methods at the same time.

In [16]:
class Spacecraft:
    def __init__(self, name, fuel):
        self.name = name
        self.__fuel = fuel                  # private

    def get_fuel(self):
        return self.__fuel

    def use_fuel(self, amount):
        if amount <= self.__fuel:
            self.__fuel = self.__fuel - amount
        else:
            print("Not enough fuel for", self.name)

    def __str__(self):
        return self.name + " (" + str(self.__fuel) + " units)"

    def __lt__(self, other):
        return self.get_fuel() < other.get_fuel()


class Probe(Spacecraft):
    def __init__(self, name, fuel, target):
        super().__init__(name, fuel)
        self.target = target

    def __str__(self):
        return super().__str__() + " heading to " + self.target


fleet = [Spacecraft("Voyager", 40),
         Probe("Juno", 90, "Jupiter"),
         Probe("Cassini", 25, "Saturn")]

for craft in fleet:
    print(craft)

print()
fleet[0].use_fuel(100)
fleet[1].use_fuel(30)
print(fleet[1])

print()
print("Sorted by fuel:")
for craft in sorted(fleet):
    print(craft)

Voyager (40 units)
Juno (90 units) heading to Jupiter
Cassini (25 units) heading to Saturn

Not enough fuel for Voyager
Juno (60 units) heading to Jupiter

Sorted by fuel:
Cassini (25 units) heading to Saturn
Voyager (40 units)
Juno (60 units) heading to Jupiter


### Summary
- **Encapsulation** keeps data inside the object and controls access to it. `name` is public, `_name` is protected by convention, and `__name` is private through **name mangling**.
- **Getters** and **setters** let the object check a value before accepting it.
- **Magic methods** have double underscores on both sides and are called by Python when you use ordinary syntax such as `print()`, `+`, `len()`, `<`, `in` and `==`.
- Writing them makes your own objects behave like Python's built-in types.
- An **inner class** is a class defined inside another, used when it only makes sense as part of the outer class.

### The Four Ideas of OOP
| Idea | What it means | Where you met it |
|---|---|---|
| **Abstraction** | hide the complicated details behind a simple name | every method you call |
| **Encapsulation** | keep data inside the object and guard it | Part 3 |
| **Inheritance** | a child class reuses a parent's work | Part 2 |
| **Polymorphism** | the same method name behaves differently per object | Part 2 |